# TP 8 — Naive Bayes
**Machine Learning · Apprentissage supervisé · Classification probabiliste**


| Exemple | Diapos | Problème |
|---|---|---|
| Exemple 1 | 60 | Catégoriel : va-t-on jouer au tennis le jour J9 ? |
| Exemple 2 | 61 | Catégoriel avec lissage : et le jour J10 ? |
| Exemple 3 | 62 | Gaussien : joue-t-on si l’humidité vaut 72 % ? |
| Exemple 4 | 63 | Multinomial : l’e-mail « gratuit gagner réunion » est-il un spam ? |

**Objectifs** : programmer **en Python pur** le comptage, les indices, le lissage, la courbe en cloche et le score ; appliquer ces fonctions aux exemples des diapos ; puis refaire les mêmes exemples avec **scikit-learn** (`CategoricalNB`, `GaussianNB`, `MultinomialNB`) et comparer les deux implémentations.

## Rappel de cours

$$P(C \mid x) \propto \underbrace{P(C)}_{\text{départ}} \times \underbrace{P(x_1 \mid C)\times P(x_2 \mid C)\times \dots}_{\text{indices}}$$

- **Départ** : part de chaque classe dans les données.
- **Indice** : probabilité d’une valeur dans une classe (on **compte**).
- Hypothèse **naïve** : les variables sont indépendantes une fois la classe connue, donc on **multiplie** les indices.
- **Lissage** (α = 1) : indice $= \dfrac{\text{comptage} + 1}{\text{total} + \text{nb de valeurs possibles}}$, pour qu’un seul zéro n’annule pas tout le produit.
- **Pourcentages** : on divise chaque score par la somme des scores.
- **Gaussien** (variable numérique) : indice = hauteur de la courbe en cloche $\mathcal N(\mu, \sigma)$ de la classe au point $x$.
- **Multinomial** (texte) : indice d’un mot $= \dfrac{\text{apparitions du mot} + 1}{\text{total des mots de la classe} + V}$.

In [1]:
import math

---
# Partie A — Implémentation from scratch (Python pur)

## Exemple 1 : le jour J9 (diapo 60)

Mêmes 8 jours que l’arbre de décision ; humidité codée **Haute** (≥ 80 %) ou **Normale**.
J9 : Ciel = Soleil, Humidité = Normale, Vent = Faible.

In [2]:
jours = [  # (ciel, humidité en %, vent, on joue ?)
    ("Soleil", 65, "Faible", "Oui"), ("Soleil", 90, "Faible", "Non"),
    ("Pluie", 70, "Faible", "Oui"),  ("Soleil", 85, "Faible", "Non"),
    ("Pluie", 80, "Fort", "Non"),    ("Soleil", 70, "Fort", "Non"),
    ("Pluie", 60, "Faible", "Oui"),  ("Soleil", 75, "Faible", "Oui"),
]
donnees = [{"ciel": c, "humidite": "Haute" if h >= 80 else "Normale", "vent": v, "classe": y} for c, h, v, y in jours]
variables = ["ciel", "humidite", "vent"]
valeurs_possibles = {"ciel": ["Soleil", "Pluie"], "humidite": ["Normale", "Haute"], "vent": ["Faible", "Fort"]}
classes = ["Oui", "Non"]

### Question 1 — Départ et indices (sans lissage)
**Réponse.** Départ : Oui = Non = 4/8 = 0,5. Indices (diapo 60) :

| Variable | Valeur | chez « Oui » | chez « Non » |
|---|---|---|---|
| Ciel | Soleil | 2/4 = 0,50 | 3/4 = 0,75 |
| Humidité | Normale | 4/4 = 1,00 | 1/4 = 0,25 |
| Vent | Faible | 4/4 = 1,00 | 2/4 = 0,50 |

In [3]:
def depart(donnees, c):
    return sum(d["classe"] == c for d in donnees) / len(donnees)

def indice(donnees, variable, valeur, c, alpha=0):
    dans_classe = [d for d in donnees if d["classe"] == c]
    comptage = sum(d[variable] == valeur for d in dans_classe)
    k = len(valeurs_possibles[variable])
    return (comptage + alpha) / (len(dans_classe) + alpha * k)

for c in classes:
    print(f"Départ {c} = {depart(donnees, c)}")
for var in variables:
    for val in valeurs_possibles[var]:
        print(f"{var:<9} {val:<8} Oui : {indice(donnees, var, val, 'Oui'):.2f}   Non : {indice(donnees, var, val, 'Non'):.2f}")
print(f"P(Soleil | Non) = {indice(donnees, "ciel", "Soleil", "Non"):.4g}")
print(f"P(Normale | Non) = {indice(donnees, "humidite", "Normale", "Non"):.4g}")
print(f"P(Haute | Oui) = {indice(donnees, "humidite", "Haute", "Oui"):.4g}")

Départ Oui = 0.5
Départ Non = 0.5
ciel      Soleil   Oui : 0.50   Non : 0.75
ciel      Pluie    Oui : 0.50   Non : 0.25
humidite  Normale  Oui : 1.00   Non : 0.25
humidite  Haute    Oui : 0.00   Non : 0.75
vent      Faible   Oui : 1.00   Non : 0.50
vent      Fort     Oui : 0.00   Non : 0.50
P(Soleil | Non) = 0.75
P(Normale | Non) = 0.25
P(Haute | Oui) = 0


### Question 2 — Score de chaque classe pour J9, puis pourcentages
**Réponse.**
- Oui : $0{,}5 \times 0{,}50 \times 1{,}00 \times 1{,}00 = 0{,}250$
- Non : $0{,}5 \times 0{,}75 \times 0{,}25 \times 0{,}50 = 0{,}047$
- Oui : $0{,}250 / (0{,}250 + 0{,}047) = 0{,}84$ → **J9 : Oui (84 %)**

In [4]:
def scores(donnees, jour, alpha=0):
    s = {}
    for c in classes:
        s[c] = depart(donnees, c)
        for var in variables:
            s[c] *= indice(donnees, var, jour[var], c, alpha)
    total = sum(s.values())
    return s, {c: (s[c] / total if total > 0 else float("nan")) for c in classes}

J9 = {"ciel": "Soleil", "humidite": "Normale", "vent": "Faible"}
s9, p9 = scores(donnees, J9)
print("Scores :", {c: round(v, 4) for c, v in s9.items()}, "| pourcentages :", {c: round(v, 3) for c, v in p9.items()})
print(f"score Oui (J9) = {s9["Oui"]:.4g}")
print(f"score Non (J9) = {s9["Non"]:.4g}")
print(f"P(Oui | J9) = {p9["Oui"]:.4g}")

Scores : {'Oui': 0.25, 'Non': 0.0469} | pourcentages : {'Oui': 0.842, 'Non': 0.158}
score Oui (J9) = 0.25
score Non (J9) = 0.04688
P(Oui | J9) = 0.8421


## Exemple 2 : le jour J10 et le lissage (diapo 61)

J10 : Ciel = Pluie, Humidité = **Haute**, Vent = Faible.

### Question 3 — Sans lissage, un zéro écrase tout
**Réponse.** Aucun jour « Oui » n’a eu une humidité haute : $P(\text{Haute} \mid \text{Oui}) = 0/4 = 0$, donc le score de « Oui » vaut 0, quoi qu’il arrive : « Non » à 100 %.

In [5]:
J10 = {"ciel": "Pluie", "humidite": "Haute", "vent": "Faible"}
s10, p10 = scores(donnees, J10)
print("Sans lissage :", {c: round(v, 4) for c, v in s10.items()}, "→", {c: round(v, 3) for c, v in p10.items()})
print(f"score Oui sans lissage = {s10["Oui"]:.4g}")
print(f"score Non sans lissage = {s10["Non"]:.4g}")

Sans lissage : {'Oui': 0.0, 'Non': 0.0469} → {'Oui': 0.0, 'Non': 1.0}
score Oui sans lissage = 0
score Non sans lissage = 0.04688


### Question 4 — Avec lissage (+1 à chaque comptage, +2 à chaque total)
**Réponse.**
- Haute chez « Oui » : $(0 + 1)/(4 + 2) = 1/6 \approx 0{,}17$ au lieu de 0
- Oui : $0{,}5 \times 3/6 \times 1/6 \times 5/6 = 0{,}035$ ; Non : $0{,}5 \times 2/6 \times 4/6 \times 3/6 = 0{,}056$
- Oui : $0{,}035/(0{,}035 + 0{,}056) = 0{,}38$ → **J10 : Non (62 %)**
- Contrôle J9 lissé : $0{,}174/(0{,}174 + 0{,}056) = 0{,}76$ → toujours « Oui »

In [6]:
print(f"P(Haute | Oui) lissé = {indice(donnees, "humidite", "Haute", "Oui", alpha=1):.4g}")
s10l, p10l = scores(donnees, J10, alpha=1)
print(f"score Oui lissé (J10) = {s10l["Oui"]:.4g}")
print(f"score Non lissé (J10) = {s10l["Non"]:.4g}")
print(f"P(Non | J10) lissé = {p10l["Non"]:.4g}")
s9l, p9l = scores(donnees, J9, alpha=1)
print(f"score Oui lissé (J9) = {s9l["Oui"]:.4g}")
print(f"P(Oui | J9) lissé = {p9l["Oui"]:.4g}")

P(Haute | Oui) lissé = 0.1667
score Oui lissé (J10) = 0.03472
score Non lissé (J10) = 0.05556
P(Non | J10) lissé = 0.6154
score Oui lissé (J9) = 0.1736
P(Oui | J9) lissé = 0.7576


## Exemple 3 : variable continue, Naive Bayes gaussien (diapo 62)

L’humidité est gardée en nombre. Question : joue-t-on si l’humidité vaut **72 %** ?

### Question 5 — Apprendre μ et σ dans chaque classe
**Réponse.**
- Oui (65, 70, 60, 75) : $\mu = 67{,}5$ ; variance $= 125/4 = 31{,}25$ → $\sigma = 5{,}59$
- Non (90, 85, 80, 70) : $\mu = 81{,}25$ ; variance $= 218{,}75/4 = 54{,}69$ → $\sigma = 7{,}40$

(La variance est la moyenne des carrés des écarts, divisée par $n$ et non par $n - 1$, comme dans scikit-learn.)

In [7]:
humidites = {c: [h for _, h, _, y in jours if y == c] for c in classes}
mu, sigma = {}, {}
for c in classes:
    v = humidites[c]
    mu[c] = sum(v) / len(v)
    sigma[c] = math.sqrt(sum((x - mu[c]) ** 2 for x in v) / len(v))
    print(f"{c} : humidités {v} → μ = {mu[c]:.2f} ; σ = {sigma[c]:.2f}")
print(f"μ Oui = {mu["Oui"]:.4g}"); print(f"σ Oui = {sigma["Oui"]:.4g}")
print(f"μ Non = {mu["Non"]:.4g}"); print(f"σ Non = {sigma["Non"]:.4g}")

Oui : humidités [65, 70, 60, 75] → μ = 67.50 ; σ = 5.59
Non : humidités [90, 85, 80, 70] → μ = 81.25 ; σ = 7.40
μ Oui = 67.5
σ Oui = 5.59
μ Non = 81.25
σ Non = 7.395


### Question 6 — Hauteur de la courbe en cloche en x = 72, puis pourcentages
**Réponse.** $\mathcal N(x ; \mu, \sigma) = \dfrac{1}{\sigma\sqrt{2\pi}}\,e^{-\frac{(x-\mu)^2}{2\sigma^2}}$
- Oui : hauteur 0,0516 ; Non : hauteur 0,0247
- Scores : Oui $0{,}5 \times 0{,}0516 = 0{,}0258$ ; Non $0{,}0123$ → **Oui (68 %)**
- Variables mixtes (+ Ciel = Soleil, Vent = Faible, indices sans lissage) : Oui $0{,}0129$ ; Non $0{,}0046$ → Oui 74 %

In [8]:
def gauss(x, mu, sigma):
    return math.exp(-(x - mu) ** 2 / (2 * sigma ** 2)) / (sigma * math.sqrt(2 * math.pi))

h = {c: gauss(72, mu[c], sigma[c]) for c in classes}
print(f"hauteur Oui en 72 = {h["Oui"]:.4g}")
print(f"hauteur Non en 72 = {h["Non"]:.4g}")
sg = {c: 0.5 * h[c] for c in classes}
p_g = sg["Oui"] / (sg["Oui"] + sg["Non"])
print(f"P(Oui | 72 %) = {p_g:.4g}")

sm = {c: sg[c] * indice(donnees, "ciel", "Soleil", c) * indice(donnees, "vent", "Faible", c) for c in classes}
print(f"score mixte Oui = {sm["Oui"]:.4g}")
print(f"score mixte Non = {sm["Non"]:.4g}")
print(f"P(Oui) mixte = {sm["Oui"] / (sm["Oui"] + sm["Non"]):.4g}")

hauteur Oui en 72 = 0.05161
hauteur Non en 72 = 0.02467
P(Oui | 72 %) = 0.6766
score mixte Oui = 0.0129
score mixte Non = 0.004626
P(Oui) mixte = 0.7361


## Exemple 4 : texte, Naive Bayes multinomial (diapo 63)

| E-mail | gratuit | gagner | réunion | rapport | Classe |
|---|---|---|---|---|---|
| E1 « gratuit gratuit gagner » | 2 | 1 | 0 | 0 | Spam |
| E2 « gratuit gagner » | 1 | 1 | 0 | 0 | Spam |
| E3 « réunion réunion rapport » | 0 | 0 | 2 | 1 | Non |
| E4 « gratuit réunion rapport » | 1 | 0 | 1 | 1 | Non |

### Question 7 — L’e-mail « gratuit gagner réunion » est-il un spam ?
**Réponse.**
- Spam (5 mots au total) : gratuit 4/9 · gagner 3/9 · réunion 1/9
- Non (6 mots au total) : gratuit 2/10 · gagner 1/10 · réunion 4/10
- Spam : $0{,}5 \times 4/9 \times 3/9 \times 1/9 = 0{,}0082$ ; Non : $0{,}5 \times 0{,}2 \times 0{,}1 \times 0{,}4 = 0{,}0040$ → **Spam (67 %)**

> **Remarque.** Sans lissage, « réunion » n’apparaît jamais dans les spams (0/5), donc le score Spam vaut 0. Mais « gagner » n’apparaît jamais non plus dans les e-mails normaux (0/6), donc le score Non vaut **aussi** 0 : on obtient 0/0 et on ne peut pas conclure. Le lissage est indispensable ici.

In [9]:
emails = ["gratuit gratuit gagner", "gratuit gagner", "réunion réunion rapport", "gratuit réunion rapport"]
etiquettes = ["Spam", "Spam", "Non", "Non"]
vocabulaire = ["gratuit", "gagner", "réunion", "rapport"]

def comptages(texte):
    mots = texte.split()
    return [mots.count(m) for m in vocabulaire]

def multinomial(emails, etiquettes, nouveau, alpha=1):
    V = len(vocabulaire)
    scores = {}
    for c in ["Spam", "Non"]:
        docs = [comptages(e) for e, y in zip(emails, etiquettes) if y == c]
        somme = [sum(d[j] for d in docs) for j in range(V)]         # apparitions de chaque mot dans la classe
        total = sum(somme)
        score = len(docs) / len(emails)                               # départ
        for mot in nouveau.split():
            score *= (somme[vocabulaire.index(mot)] + alpha) / (total + alpha * V)
        scores[c] = score
    total_scores = sum(scores.values())
    if total_scores == 0:
        return scores, {c: float("nan") for c in scores}       # 0 / 0 : impossible de conclure
    return scores, {c: s / total_scores for c, s in scores.items()}

s_txt, p_txt = multinomial(emails, etiquettes, "gratuit gagner réunion")
print(f"score Spam = {s_txt["Spam"]:.4g}")
print(f"score Non = {s_txt["Non"]:.4g}")
print(f"P(Spam) = {p_txt["Spam"]:.4g}")
s0, p0 = multinomial(emails, etiquettes, "gratuit gagner réunion", alpha=0)
print("Sans lissage : scores =", s0, "→ les DEUX scores valent 0 : impossible de conclure")

score Spam = 0.00823
score Non = 0.004
P(Spam) = 0.6729
Sans lissage : scores = {'Spam': 0.0, 'Non': 0.0} → les DEUX scores valent 0 : impossible de conclure


---
# Partie B — Les mêmes exemples avec scikit-learn

### Question 8 — `CategoricalNB`, `GaussianNB`, `MultinomialNB`
**Réponse.**
- `CategoricalNB(alpha=1.0)` fait exactement notre lissage. Pour reproduire le calcul **sans** lissage (J9), on prend `alpha` quasi nul avec `force_alpha=True`. Les catégories doivent être codées en nombres (`OrdinalEncoder`).
- `GaussianNB()` apprend μ (`theta_`) et σ² (`var_`) dans chaque classe.
- `MultinomialNB(alpha=1.0)` travaille sur les comptages de mots produits par `CountVectorizer`.

In [10]:
import numpy as np, pandas as pd
from sklearn.naive_bayes import CategoricalNB, GaussianNB, MultinomialNB
from sklearn.preprocessing import OrdinalEncoder
from sklearn.feature_extraction.text import CountVectorizer

df = pd.DataFrame(donnees)
enc = OrdinalEncoder(categories=[valeurs_possibles[v] for v in variables])
Xc = enc.fit_transform(df[variables]); yc = df["classe"]
x9 = enc.transform(pd.DataFrame([J9])); x10 = enc.transform(pd.DataFrame([J10]))

nb_sans = CategoricalNB(alpha=1e-10, force_alpha=True).fit(Xc, yc)
nb_lisse = CategoricalNB(alpha=1.0).fit(Xc, yc)
p9_sk = dict(zip(nb_sans.classes_, nb_sans.predict_proba(x9)[0]))
p10_sk = dict(zip(nb_lisse.classes_, nb_lisse.predict_proba(x10)[0]))
p9l_sk = dict(zip(nb_lisse.classes_, nb_lisse.predict_proba(x9)[0]))
print("CategoricalNB J9 sans lissage :", {str(k): round(float(v), 3) for k, v in p9_sk.items()})
print("CategoricalNB J10 avec lissage :", {str(k): round(float(v), 3) for k, v in p10_sk.items()})
print("CategoricalNB J9 avec lissage  :", {str(k): round(float(v), 3) for k, v in p9l_sk.items()})

H = np.array([h for _, h, _, _ in jours]).reshape(-1, 1)
gnb = GaussianNB().fit(H, [y for *_, y in jours])
print("\nGaussianNB : classes", gnb.classes_, "μ =", gnb.theta_.ravel(), "σ =", np.sqrt(gnb.var_.ravel()).round(2))
p72_sk = dict(zip(gnb.classes_, gnb.predict_proba([[72]])[0]))
print("GaussianNB P(classe | 72 %) :", {str(k): round(float(v), 3) for k, v in p72_sk.items()})

cv = CountVectorizer(vocabulary=vocabulaire)
mnb = MultinomialNB(alpha=1.0).fit(cv.transform(emails), etiquettes)
pspam_sk = dict(zip(mnb.classes_, mnb.predict_proba(cv.transform(["gratuit gagner réunion"]))[0]))
print("\nMultinomialNB :", {str(k): round(float(v), 3) for k, v in pspam_sk.items()})

CategoricalNB J9 sans lissage : {'Non': 0.158, 'Oui': 0.842}
CategoricalNB J10 avec lissage : {'Non': 0.615, 'Oui': 0.385}
CategoricalNB J9 avec lissage  : {'Non': 0.242, 'Oui': 0.758}

GaussianNB : classes ['Non' 'Oui'] μ = [81.25 67.5 ] σ = [7.4  5.59]
GaussianNB P(classe | 72 %) : {'Non': 0.323, 'Oui': 0.677}

MultinomialNB : {'Non': 0.327, 'Spam': 0.673}


---
# Partie C — Comparaison et validation
### Question 9 — Les résultats concordent-ils ?

In [12]:
lignes = [
    ("J9 P(Oui) sans lissage", p9["Oui"], p9_sk["Oui"]),
    ("J10 P(Non) avec lissage", p10l["Non"], p10_sk["Non"]),
    ("J9 P(Oui) avec lissage", p9l["Oui"], p9l_sk["Oui"]),
    ("Gaussien μ Oui", mu["Oui"], gnb.theta_[list(gnb.classes_).index("Oui"), 0]),
    ("Gaussien σ Oui", sigma["Oui"], np.sqrt(gnb.var_[list(gnb.classes_).index("Oui"), 0])),
    ("Gaussien P(Oui | 72 %)", p_g, p72_sk["Oui"]),
    ("Texte P(Spam)", p_txt["Spam"], pspam_sk["Spam"]),
]
tab = pd.DataFrame(lignes, columns=["Grandeur", "Python pur", "scikit-learn"])
display(tab.round(4))
assert np.allclose(tab["Python pur"], tab["scikit-learn"], atol=1e-6)
print(" Python pur = scikit-learn pour les trois variantes de Naive Bayes")

,Grandeur,Python pur,scikit-learn
0,J9 P(Oui) sans lissage,0.8421,0.8421
1,J10 P(Non) avec lissage,0.6154,0.6154
2,J9 P(Oui) avec lissage,0.7576,0.7576
3,Gaussien μ Oui,67.5000,67.5000
4,Gaussien σ Oui,5.5902,5.5902
5,Gaussien P(Oui | 72 %),0.6766,0.6766
6,Texte P(Spam),0.6729,0.6729


 Python pur = scikit-learn pour les trois variantes de Naive Bayes


## Conclusion


**À retenir**
- Apprendre = **compter** : un seul passage sur les données, sans descente de gradient.
- Le **lissage** évite qu’une valeur jamais vue annule tout le produit.
- Catégories → `CategoricalNB` ; nombres → `GaussianNB` ; comptages de mots → `MultinomialNB`.